# Step 5B: Calculate LST Zonal Statistics

## 100m

In [1]:
# # Step 5B: LST Zonal Statistics -> single JSON (+ one reference grid geometry file)
# #
# # Reads the SAME Step 4 manifest that Step 5A consumes
# # (`step4_merge_manifest.json` -- the merged/pass-through masked tiffs),
# # rebuilds the identical spatial grid + grid-id raster used in Step 5A (so
# # grid_id values match across outputs), and computes per-grid-cell zonal
# # statistics (mean, median, std) of LST for every tiff.
# #
# # Rather than writing a full grid geometry into a GPKG per scene (wasteful --
# # the geometry is identical across every scene), the grid geometry is written
# # ONCE to a reference file, and all per-scene stats are stored as arrays
# # (positionally aligned to a shared `grid_ids` list) inside one JSON.
# #
# # Output:
# #   <root_dir>/lst_zonal_stats/grid_reference.gpkg        (geometry, written once)
# #   <root_dir>/lst_zonal_stats/lst_zonal_stats.json        (all scenes' stats)
#
# import os
# import json
# import threading
# import numpy as np
# import pandas as pd
# import geopandas as gpd
# import rasterio
# import shapely
#
# from datetime import datetime, timedelta
# from zoneinfo import ZoneInfo
# from concurrent.futures import ThreadPoolExecutor, as_completed
#
# from rasterio.features import rasterize
# from rasterio.warp import reproject, Resampling, transform_bounds
# from rasterio.transform import from_origin
#
#
# # ================== Load Config ==================
#
# CONFIG_PATH = "config.json"
#
# with open(CONFIG_PATH) as f:
#     config = json.load(f)
#
# grid_cfg = config["grid_coverage"]
# out_cfg = config["output"]
# proc_cfg = config["processing"]
#
# zonal_cfg = config.get("zonal_stats", {})
#
# MAX_WORKERS = proc_cfg.get("zonal_stats_max_workers", 10)
#
# GRID_CELL_SIZE_M = grid_cfg["grid_cell_size_m"]
# GRID_ID_FIELD = grid_cfg["grid_id_field"]
# CLIP_GRID = grid_cfg["clip_grid_to_boundary"]
# BOUNDARY = grid_cfg["study_region_shapefile"]
# EXISTING_GRID = grid_cfg.get("existing_grid_shapefile")
#
# DAY_START_HOUR = zonal_cfg.get("day_start_hour", 6)
# DAY_END_HOUR = zonal_cfg.get("day_end_hour", 18)
#
# # Decimal places to round stats to before writing JSON (keeps file size sane;
# # set to None in code below to disable rounding entirely)
# ROUND_DECIMALS = zonal_cfg.get("round_decimals", 3)
#
# root_dir = out_cfg.get("root_dir") or os.path.dirname(os.path.normpath(config["input_folder"]))
# masked_dir = os.path.join(root_dir, out_cfg.get("masked_tiff_dirname", "masked_tiff"))
# step4_path = os.path.join(masked_dir, "step4_merge_manifest.json")
#
# with open(step4_path) as f:
#     step4 = json.load(f)
#
# hour_bins_in = step4["hour_bins"]
# HOURS = [f"{i:02d}" for i in range(24)]
#
# zonal_stats_dir = os.path.join(root_dir, out_cfg.get("zonal_stats_dirname", "lst_zonal_stats"))
# os.makedirs(zonal_stats_dir, exist_ok=True)
#
# print(f"Loaded Step 4 manifest -> {step4_path}")
# print(f"Zonal stats workers: {MAX_WORKERS}")
#
#
# # ================== Build Spatial Grid (identical to Step 5A) ==================
#
# def build_grid(boundary_shp, cell_size, grid_id, clip):
#     gdf = gpd.read_file(boundary_shp)
#
#     if gdf.crs.is_geographic:
#         gdf = gdf.to_crs(gdf.estimate_utm_crs())
#
#     boundary = gdf.geometry.union_all()
#     minx, miny, maxx, maxy = boundary.bounds
#
#     xs = np.arange(minx, maxx + cell_size, cell_size)
#     ys = np.arange(miny, maxy + cell_size, cell_size)
#     xx, yy = np.meshgrid(xs[:-1], ys[:-1])
#
#     cells = shapely.box(xx.ravel(), yy.ravel(), xx.ravel() + cell_size, yy.ravel() + cell_size)
#     cells = cells[shapely.intersects(cells, boundary)]
#
#     if clip:
#         cells = shapely.intersection(cells, boundary)
#         cells = cells[~shapely.is_empty(cells)]
#
#     return gpd.GeoDataFrame(
#         {grid_id: range(1, len(cells) + 1)},
#         geometry=cells,
#         crs=gdf.crs,
#     )
#
#
# def load_grid():
#     if EXISTING_GRID and os.path.isfile(EXISTING_GRID):
#         return gpd.read_file(EXISTING_GRID)
#     return build_grid(BOUNDARY, GRID_CELL_SIZE_M, GRID_ID_FIELD, CLIP_GRID)
#
#
# grid = load_grid()
# print(f"Grid cells: {len(grid)}")
#
#
# # ================== Write reference grid geometry ONCE ==================
# # All scenes share this same geometry -- no need to repeat it per scene.
#
# grid_reference_path = os.path.join(zonal_stats_dir, "grid_reference.gpkg")
# grid[[GRID_ID_FIELD, "geometry"]].to_file(grid_reference_path, driver="GPKG")
# print(f"Grid reference geometry saved -> {grid_reference_path}")
#
#
# # ================== Master Raster Grid + Grid-ID Raster (identical to Step 5A) ==================
#
# def create_master_grid(grid, resolution):
#     minx, miny, maxx, maxy = grid.total_bounds
#     width = int(np.ceil((maxx - minx) / resolution))
#     height = int(np.ceil((maxy - miny) / resolution))
#     transform = from_origin(minx, maxy, resolution, resolution)
#     return {"height": height, "width": width, "transform": transform, "crs": grid.crs}
#
#
# master_grid = create_master_grid(grid, GRID_CELL_SIZE_M)
# print("Master raster:", master_grid["height"], "x", master_grid["width"])
#
#
# def build_grid_id_raster(grid, master):
#     grid_proj = grid.to_crs(master["crs"])
#     shapes = ((geom, i + 1) for i, geom in enumerate(grid_proj.geometry))
#     return rasterize(
#         shapes,
#         out_shape=(master["height"], master["width"]),
#         transform=master["transform"],
#         fill=0,
#         dtype="int32",
#     )
#
#
# grid_ids = build_grid_id_raster(grid, master_grid)
# N_GRID_CELLS = len(grid)
# GRID_ID_VALUES = grid[GRID_ID_FIELD].to_numpy()
#
# FLAT_GRID_IDS = grid_ids.ravel()
#
# print("Grid ID raster:", grid_ids.shape)
#
#
# # ================== Align TIFF to Master Grid (identical to Step 5A) ==================
#
# def align_tiff(raster_path, master):
#     with rasterio.open(raster_path) as src:
#         data = src.read(1).astype("float32")
#
#         if src.nodata is not None:
#             data[data == src.nodata] = np.nan
#
#         aligned = np.full((master["height"], master["width"]), np.nan, dtype="float32")
#
#         if src.crs is None or src.transform == rasterio.Affine.identity():
#             print(f"\nWARNING: Missing georeferencing\n{raster_path}")
#
#         reproject(
#             source=data,
#             destination=aligned,
#             src_transform=src.transform,
#             src_crs=src.crs,
#             dst_transform=master["transform"],
#             dst_crs=master["crs"],
#             src_nodata=np.nan,
#             dst_nodata=np.nan,
#             resampling=Resampling.nearest,
#         )
#
#         src_crs = src.crs
#         src_bounds = src.bounds
#
#     return aligned, src_crs, src_bounds
#
#
# # ================== Local Solar Time + Day/Night ==================
#
# def get_centroid_lon(src_crs, src_bounds):
#     lon_min, lat_min, lon_max, lat_max = transform_bounds(
#         src_crs, "EPSG:4326", *src_bounds
#     )
#     return (lon_min + lon_max) / 2.0
#
#
# def compute_local_solar_time(dt_utc, centroid_lon):
#     offset_hours = centroid_lon / 15.0
#     return dt_utc + timedelta(hours=offset_hours)
#
#
# def classify_day_night(solar_hour, day_start=DAY_START_HOUR, day_end=DAY_END_HOUR):
#     return "day" if day_start <= solar_hour < day_end else "night"
#
#
# # ================== Per-cell Zonal Stats (mean, median, std) ==================
#
# def compute_zonal_stats(data):
#     """Groups aligned raster values by grid_id and returns arrays (length
#     N_GRID_CELLS, positionally aligned with GRID_ID_VALUES) of
#     mean, median, std, and valid-pixel count."""
#     flat_vals = data.ravel()
#     valid = np.isfinite(flat_vals) & (FLAT_GRID_IDS > 0)
#
#     ids = FLAT_GRID_IDS[valid]
#     vals = flat_vals[valid]
#
#     df = pd.DataFrame({"cell_idx": ids, "value": vals})
#     grouped = df.groupby("cell_idx")["value"].agg(["mean", "median", "std", "count"])
#     grouped = grouped.reindex(range(1, N_GRID_CELLS + 1))
#
#     return (
#         grouped["mean"].to_numpy(),
#         grouped["median"].to_numpy(),
#         grouped["std"].to_numpy(),
#         grouped["count"].fillna(0).to_numpy(),
#     )
#
#
# def array_to_list(arr, decimals=ROUND_DECIMALS):
#     """NaN-safe float32 array -> plain python list, optionally rounded."""
#     arr = arr.astype("float64")  # avoid float32 json serialization quirks
#     if decimals is not None:
#         arr = np.round(arr, decimals)
#     return [None if np.isnan(v) else v for v in arr]
#
#
# # ================== Per-tiff Worker ==================
#
# def process_tiff(hour, entry):
#     tiff_path = entry["tiff_path"]
#     tiff_id = os.path.splitext(os.path.basename(tiff_path))[0]
#
#     data, src_crs, src_bounds = align_tiff(tiff_path, master_grid)
#     lst_mean, lst_median, lst_std, n_valid = compute_zonal_stats(data)
#
#     dt_local = datetime.fromisoformat(entry["datetime_local"])
#     dt_utc = dt_local.astimezone(ZoneInfo("UTC"))
#
#     centroid_lon = get_centroid_lon(src_crs, src_bounds)
#     solar_dt = compute_local_solar_time(dt_utc, centroid_lon)
#     day_night = classify_day_night(solar_dt.hour + solar_dt.minute / 60.0)
#     season = entry.get("season", "Unknown")
#     acquisition_date = dt_local.date().isoformat()
#
#     scene_record = {
#         "tiff_id": tiff_id,
#         "source_tiff_path": tiff_path,
#         "hour": hour,
#         "acquisition_date": acquisition_date,
#         "datetime_local": entry["datetime_local"],
#         "local_solar_time": solar_dt.isoformat(),
#         "day_night": day_night,
#         "season": season,
#         "is_merged": entry.get("is_merged", False),
#         "n_grid_cells_with_data": int((n_valid > 0).sum()),
#         # Arrays below are positionally aligned to top-level "grid_ids"
#         "lst_mean": array_to_list(lst_mean),
#         "lst_median": array_to_list(lst_median),
#         "lst_std": array_to_list(lst_std),
#         "n_valid_pixels": [int(v) for v in n_valid],
#     }
#
#     return tiff_id, scene_record
#
#
# # ================== Run in Parallel ==================
#
# jobs = [(h, e) for h in HOURS for e in hour_bins_in.get(h, [])]
# print(f"Tiffs to process: {len(jobs)}")
#
# scene_records = {}
# errors = []
# results_lock = threading.Lock()
#
# with ThreadPoolExecutor(MAX_WORKERS) as ex:
#     futures = {ex.submit(process_tiff, h, e): (h, e) for h, e in jobs}
#
#     for i, future in enumerate(as_completed(futures), 1):
#         h, e = futures[future]
#         try:
#             tiff_id, scene_record = future.result()
#             with results_lock:
#                 scene_records[tiff_id] = scene_record
#         except Exception as err:
#             errors.append({"hour": h, "tiff_path": e["tiff_path"], "error": str(err)})
#
#         if i % 10 == 0 or i == len(jobs):
#             print(f"Processed {i}/{len(jobs)} | errors so far: {len(errors)}")
#
# print(f"\nScenes processed: {len(scene_records)}")
# print(f"Errors: {len(errors)}")
#
# if errors:
#     print("\n=== Zonal Stats Errors ===")
#     for e in errors:
#         print(f"\nHour: {e['hour']}")
#         print(f"TIFF: {e['tiff_path']}")
#         print(f"Error: {e['error']}")
#
#
# # ================== Write single consolidated JSON ==================
#
# manifest = {
#     "step": "step5b_zonal_stats",
#     "generated_at": datetime.now(ZoneInfo("UTC")).isoformat(),
#     "grid_reference_path": grid_reference_path,
#     "grid_id_field": GRID_ID_FIELD,
#     "grid_ids": GRID_ID_VALUES.tolist(),  # canonical order -- every scene's arrays align to this
#     "n_grid_cells": N_GRID_CELLS,
#     "day_start_hour": DAY_START_HOUR,
#     "day_end_hour": DAY_END_HOUR,
#     "round_decimals": ROUND_DECIMALS,
#     "n_scenes": len(scene_records),
#     "n_errors": len(errors),
#     "errors": errors,
#     "scenes": sorted(scene_records.values(), key=lambda e: e["datetime_local"]),
# }
#
# zonal_stats_json_path = os.path.join(zonal_stats_dir, "lst_zonal_stats.json")
#
# print("\nStarting json dump")
# with open(zonal_stats_json_path, "w") as f:
#     json.dump(manifest, f, default=str)  # no indent -- keeps file size down given array volume
#
# print(f"\nStep 5B stats JSON saved -> {zonal_stats_json_path}")
# print(f"Grid reference geometry -> {grid_reference_path}")
# print(f"Total scenes: {manifest['n_scenes']}")

## 70m

In [1]:
# ============================================================
# Step 5B: LST Grid Sampling (SIMPLIFIED -- native pixel = grid cell)
# ------------------------------------------------------------
# ECO_L2T_LSTE is delivered pre-projected on a fixed 70 m UTM/MGRS
# grid. Since grid_cell_size_m == native resolution, each grid cell
# corresponds to exactly ONE native pixel -- there is nothing to
# aggregate. So instead of zonal stats (mean/median/std over a
# block of pixels), we just directly SAMPLE that one pixel's value
# per grid cell per scene. No resampling, no block-reduction, no
# std (undefined for n=1 anyway).
# Input:
#   step5_grid_coverage_manifest.json
# Only TIFFs retained by Step 5A (coverage >= threshold) are processed.
# The grid is saved as a SHAPEFILE (.shp) so it can be reused
# directly by covariate zonal-stats tools (rasterstats, QGIS,
# ArcPy, etc.) that expect standard shapefile input.
#
# Output:
#   <root_dir>/lst_zonal_stats/grid_reference.shp      (grid geometry -- for covariates)
#   <root_dir>/lst_zonal_stats/master_grid.json        (crs/transform/shape -- for covariates)
#   <root_dir>/lst_zonal_stats/master_grid_ids.tif     (grid-id raster -- for covariates)
#   <root_dir>/lst_zonal_stats/lst_zonal_stats.json    (all scenes' sampled LST values)

import os
import json
import threading
import numpy as np
import geopandas as gpd
import rasterio
import shapely

from collections import Counter
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo
from concurrent.futures import ThreadPoolExecutor, as_completed

from rasterio.features import rasterize
from rasterio.warp import reproject, Resampling, transform_bounds
from rasterio.transform import from_origin


# ================== Load Config ==================

CONFIG_PATH = "config.json"

with open(CONFIG_PATH) as f:
    config = json.load(f)

grid_cfg = config["grid_coverage"]
out_cfg = config["output"]
proc_cfg = config["processing"]
zonal_cfg = config.get("zonal_stats", {})

MAX_WORKERS = proc_cfg.get("zonal_stats_max_workers", 10)

GRID_CELL_SIZE_M = grid_cfg["grid_cell_size_m"]
GRID_ID_FIELD = grid_cfg["grid_id_field"]
CLIP_GRID = grid_cfg["clip_grid_to_boundary"]
BOUNDARY = grid_cfg["study_region_shapefile"]

DAY_START_HOUR = zonal_cfg.get("day_start_hour", 6)
DAY_END_HOUR = zonal_cfg.get("day_end_hour", 18)
ALIGN_TOL_PX = zonal_cfg.get("align_tolerance_px", 1e-3)
ROUND_DECIMALS = zonal_cfg.get("round_decimals", 3)

CONVERT_TO_CELSIUS = zonal_cfg.get("convert_to_celsius", True)
KELVIN_TO_CELSIUS_OFFSET = 273.15
LST_UNITS = "celsius" if CONVERT_TO_CELSIUS else "kelvin"

root_dir = out_cfg.get("root_dir") or os.path.dirname(os.path.normpath(config["input_folder"]))
masked_dir = os.path.join(root_dir, out_cfg.get("masked_tiff_dirname", "masked_tiff"))

step5_path = os.path.join(masked_dir, "step5_grid_coverage_manifest.json")

with open(step5_path) as f:
    step5 = json.load(f)

hour_bins_in = step5["hour_bins"]

HOURS = [f"{i:02d}" for i in range(24)]

zonal_stats_dir = os.path.join(root_dir, out_cfg.get("zonal_stats_dirname", "lst_zonal_stats"))
os.makedirs(zonal_stats_dir, exist_ok=True)

print(f"Loaded Step 4 manifest -> {step5_path}")
print(f"Workers: {MAX_WORKERS}")
print(f"LST output units: {LST_UNITS}")

jobs = [(h, e) for h in HOURS for e in hour_bins_in.get(h, [])]
print(f"Tiffs to process: {len(jobs)}")
if not jobs:
    raise ValueError("No TIFFs passed the Step 5A coverage threshold -- nothing to sample")


# ================== Probe tiff headers (metadata only) ==================

def probe_tiff(path):
    with rasterio.open(path) as src:
        return {
            "path": path,
            "crs": src.crs,
            "transform": src.transform,
            "res": (abs(src.transform.a), abs(src.transform.e)),
            "bounds": src.bounds,
            "width": src.width,
            "height": src.height,
        }

print("\nProbing tiff headers to determine native LST grid...")
probes = [probe_tiff(e["tiff_path"]) for _, e in jobs]

crs_counts = Counter(str(p["crs"]) for p in probes)
res_counts = Counter(p["res"] for p in probes)
print(f"CRS found across scenes: {dict(crs_counts)}")
print(f"Resolutions found across scenes: {dict(res_counts)}")

master_crs_str, n_master_crs = crs_counts.most_common(1)[0]
master_res, n_master_res = res_counts.most_common(1)[0]
NATIVE_RES = master_res[0]

if n_master_crs < len(probes):
    print(f"  *** NOTE: {len(probes) - n_master_crs} scene(s) not in majority CRS "
          f"({master_crs_str}) -- those will be reprojected (fallback), everyone else "
          f"sampled directly. ***")

master_crs = next(p["crs"] for p in probes if str(p["crs"]) == master_crs_str)
print(f"Master CRS: {master_crs}")
print(f"Native LST resolution: {NATIVE_RES} m")

# This simplified pipeline assumes grid cell == native pixel exactly.
# If you ever want coarser cells again, you'd need to reintroduce
# block-aggregation -- direct sampling only makes sense at native res.
if not np.isclose(GRID_CELL_SIZE_M, NATIVE_RES):
    raise ValueError(
        f"config grid_cell_size_m ({GRID_CELL_SIZE_M}) != native LST resolution "
        f"({NATIVE_RES} m). Direct pixel sampling requires grid_cell_size_m to equal "
        f"the native resolution exactly -- otherwise you need real zonal aggregation, "
        f"not a single-pixel sample."
    )
print(f"Grid cell size ({GRID_CELL_SIZE_M} m) == native resolution -- direct sampling is valid.")


# ================== Build Master Grid from the LST rasters' own footprint ==================

def bounds_in_master_crs(p):
    if str(p["crs"]) == master_crs_str:
        return p["bounds"]
    return transform_bounds(p["crs"], master_crs, *p["bounds"])

all_bounds = [bounds_in_master_crs(p) for p in probes]
minx = min(b[0] for b in all_bounds)
miny = min(b[1] for b in all_bounds)
maxx = max(b[2] for b in all_bounds)
maxy = max(b[3] for b in all_bounds)

ref_transform = next(p["transform"] for p in probes if str(p["crs"]) == master_crs_str)
ref_origin_x, ref_origin_y = ref_transform.c, ref_transform.f

def snap_down(value, origin, res):
    n = np.floor((value - origin) / res)
    return origin + n * res

def snap_up(value, origin, res):
    n = np.ceil((value - origin) / res)
    return origin + n * res

master_minx = snap_down(minx, ref_origin_x, NATIVE_RES)
master_maxy = snap_up(maxy, ref_origin_y, NATIVE_RES)
master_maxx = snap_up(maxx, ref_origin_x, NATIVE_RES)
master_miny = snap_down(miny, ref_origin_y, NATIVE_RES)

NATIVE_WIDTH = int(round((master_maxx - master_minx) / NATIVE_RES))
NATIVE_HEIGHT = int(round((master_maxy - master_miny) / NATIVE_RES))

master_transform = from_origin(master_minx, master_maxy, NATIVE_RES, NATIVE_RES)

print(f"\nMaster grid: {NATIVE_HEIGHT} x {NATIVE_WIDTH} px @ {NATIVE_RES} m "
      f"= {NATIVE_HEIGHT} x {NATIVE_WIDTH} grid cells (1:1 with pixels)")

master_grid = {
    "height": NATIVE_HEIGHT, "width": NATIVE_WIDTH,
    "transform": master_transform, "crs": master_crs,
}


# ================== Build vector grid cells (1 cell == 1 native pixel) ==================

col_idx, row_idx = np.meshgrid(np.arange(NATIVE_WIDTH), np.arange(NATIVE_HEIGHT))
col_idx = col_idx.ravel()
row_idx = row_idx.ravel()

cell_minx = master_minx + col_idx * NATIVE_RES
cell_maxy = master_maxy - row_idx * NATIVE_RES
cells = shapely.box(cell_minx, cell_maxy - NATIVE_RES, cell_minx + NATIVE_RES, cell_maxy)

grid = gpd.GeoDataFrame(
    {GRID_ID_FIELD: np.arange(1, len(cells) + 1)},
    geometry=cells,
    crs=master_crs,
)
grid["_row"] = row_idx
grid["_col"] = col_idx

if CLIP_GRID and BOUNDARY:
    boundary_gdf = gpd.read_file(BOUNDARY).to_crs(master_crs)
    boundary_geom = boundary_gdf.geometry.union_all()
    keep = shapely.intersects(grid.geometry.to_numpy(), boundary_geom)
    dropped = (~keep).sum()
    grid = grid[keep].copy()
    print(f"Clipped grid to study boundary -- dropped {dropped} cells outside boundary "
          f"({len(grid)} remain)")

print(f"Grid cells: {len(grid)}")

# Row/col of each surviving grid cell, for direct pixel indexing below.
GRID_ROW = grid["_row"].to_numpy()
GRID_COL = grid["_col"].to_numpy()
GRID_ID_VALUES = grid[GRID_ID_FIELD].to_numpy()
N_GRID_CELLS = len(grid)


# ================== Save grid geometry as SHAPEFILE (for covariates) ==================
# Shapefiles truncate field names to 10 chars and don't like a leading
# underscore in some drivers -- drop the helper columns before saving,
# they were only needed internally for pixel indexing above.

grid_reference_path = os.path.join(zonal_stats_dir, "grid_reference.shp")
grid_out = grid[[GRID_ID_FIELD, "geometry"]].copy()
if len(GRID_ID_FIELD) > 10:
    print(f"  *** NOTE: grid_id_field '{GRID_ID_FIELD}' is >10 chars -- "
          f"shapefile driver will truncate it. Consider a shorter field name "
          f"in config if this matters for downstream tools. ***")
grid_out.to_file(grid_reference_path, driver="ESRI Shapefile")
print(f"Grid reference shapefile saved -> {grid_reference_path}")


# ================== Save the MASTER GRID (for covariate alignment) ==================

master_grid_json_path = os.path.join(zonal_stats_dir, "master_grid.json")
master_grid_meta = {
    "crs_wkt": master_crs.to_wkt(),
    "transform": list(master_transform)[:6],
    "width": NATIVE_WIDTH,
    "height": NATIVE_HEIGHT,
    "resolution_m": NATIVE_RES,
    "grid_id_field": GRID_ID_FIELD,
    "bounds": [master_minx, master_miny, master_maxx, master_maxy],
    "grid_reference_path": grid_reference_path,
}
with open(master_grid_json_path, "w") as f:
    json.dump(master_grid_meta, f, indent=2)
print(f"Master grid metadata saved -> {master_grid_json_path}")

grid_id_raster_path = os.path.join(zonal_stats_dir, "master_grid_ids.tif")
grid_id_array = rasterize(
    ((geom, gid) for geom, gid in zip(grid.geometry, grid[GRID_ID_FIELD])),
    out_shape=(NATIVE_HEIGHT, NATIVE_WIDTH),
    transform=master_transform,
    fill=0,
    dtype="int32",
)
with rasterio.open(
    grid_id_raster_path, "w", driver="GTiff",
    height=NATIVE_HEIGHT, width=NATIVE_WIDTH, count=1, dtype="int32",
    crs=master_crs, transform=master_transform, nodata=0,
) as dst:
    dst.write(grid_id_array, 1)
print(f"Grid-id raster saved -> {grid_id_raster_path}")


# ================== Sample each tiff's pixels directly (NO aggregation) ==================

def sample_tiff_on_grid(raster_path, probe, master):
    """Reads the tiff and, for each grid cell, samples the single native
    pixel value at that cell's (row, col) -- no reprojection, no
    interpolation, no aggregation -- PROVIDED the tiff shares the master
    CRS and resolution (normal case). Falls back to a reproject (with a
    warning) only if that's not true for this scene, since a genuinely
    misaligned/foreign-CRS scene can't be indexed directly."""

    same_crs = str(probe["crs"]) == master_crs_str
    same_res = np.isclose(probe["res"][0], NATIVE_RES) and np.isclose(probe["res"][1], NATIVE_RES)

    if same_crs and same_res:
        t = probe["transform"]
        col_off_f = (t.c - master["transform"].c) / NATIVE_RES
        row_off_f = (master["transform"].f - t.f) / NATIVE_RES
        col_off = int(round(col_off_f))
        row_off = int(round(row_off_f))

        if (abs(col_off_f - col_off) > ALIGN_TOL_PX or
                abs(row_off_f - row_off) > ALIGN_TOL_PX):
            print(f"\n  *** WARNING: {raster_path} not on integer pixel offset "
                  f"({col_off_f - col_off:.4f}, {row_off_f - row_off:.4f} px) -- "
                  f"falling back to reproject. ***")
            same_crs = False

    if same_crs and same_res:
        with rasterio.open(raster_path) as src:
            data = src.read(1).astype("float32")
            if src.nodata is not None:
                data[data == src.nodata] = np.nan
        h, w = data.shape

        # Map each grid cell's master row/col into this scene's local row/col.
        local_row = GRID_ROW - row_off
        local_col = GRID_COL - col_off
        in_bounds = (local_row >= 0) & (local_row < h) & (local_col >= 0) & (local_col < w)

        vals = np.full(N_GRID_CELLS, np.nan, dtype="float32")
        vals[in_bounds] = data[local_row[in_bounds], local_col[in_bounds]]
        method = "direct"
    else:
        # Fallback: reproject full scene onto master grid, then sample.
        aligned = np.full((master["height"], master["width"]), np.nan, dtype="float32")
        with rasterio.open(raster_path) as src:
            data = src.read(1).astype("float32")
            if src.nodata is not None:
                data[data == src.nodata] = np.nan
            reproject(
                source=data, destination=aligned,
                src_transform=src.transform, src_crs=src.crs,
                dst_transform=master["transform"], dst_crs=master["crs"],
                src_nodata=np.nan, dst_nodata=np.nan,
                resampling=Resampling.nearest,
            )
        vals = aligned[GRID_ROW, GRID_COL]
        method = "reprojected"

    if CONVERT_TO_CELSIUS:
        vals = vals - KELVIN_TO_CELSIUS_OFFSET

    return vals, method


# ================== Local Solar Time + Day/Night ==================

def get_centroid_lon(src_crs, src_bounds):
    lon_min, lat_min, lon_max, lat_max = transform_bounds(src_crs, "EPSG:4326", *src_bounds)
    return (lon_min + lon_max) / 2.0

def compute_local_solar_time(dt_utc, centroid_lon):
    offset_hours = centroid_lon / 15.0
    return dt_utc + timedelta(hours=offset_hours)

def classify_day_night(solar_hour, day_start=DAY_START_HOUR, day_end=DAY_END_HOUR):
    return "day" if day_start <= solar_hour < day_end else "night"


def array_to_list(arr, decimals=ROUND_DECIMALS):
    arr = arr.astype("float64")
    if decimals is not None:
        arr = np.round(arr, decimals)
    return [None if np.isnan(v) else v for v in arr]


# ================== Per-tiff Worker ==================

def process_tiff(hour, entry, probe):
    tiff_path = entry["tiff_path"]
    tiff_id = os.path.splitext(os.path.basename(tiff_path))[0]

    lst_vals, method = sample_tiff_on_grid(tiff_path, probe, master_grid)

    dt_local = datetime.fromisoformat(entry["datetime_local"])
    dt_utc = dt_local.astimezone(ZoneInfo("UTC"))

    centroid_lon = get_centroid_lon(probe["crs"], probe["bounds"])
    solar_dt = compute_local_solar_time(dt_utc, centroid_lon)
    day_night = classify_day_night(solar_dt.hour + solar_dt.minute / 60.0)
    season = entry.get("season", "Unknown")
    acquisition_date = dt_local.date().isoformat()

    scene_record = {
        "tiff_id": tiff_id,
        "source_tiff_path": tiff_path,
        "hour": hour,
        "acquisition_date": acquisition_date,
        "datetime_local": entry["datetime_local"],
        "local_solar_time": solar_dt.isoformat(),
        "day_night": day_night,
        "season": season,
        "is_merged": entry.get("is_merged", False),
        "grid_placement_method": method,  # "direct" (expected) or "reprojected" (fallback)
        "n_grid_cells_with_data": int(np.isfinite(lst_vals).sum()),
        # Single sampled value per grid cell -- this IS the LST value,
        # not a "mean" over anything (BLOCK=1 -> no aggregation happened).
        "lst_value": array_to_list(lst_vals),
    }

    return tiff_id, scene_record


# ================== Run in Parallel ==================

probe_by_path = {p["path"]: p for p in probes}

scene_records = {}
errors = []
results_lock = threading.Lock()

with ThreadPoolExecutor(MAX_WORKERS) as ex:
    futures = {
        ex.submit(process_tiff, h, e, probe_by_path[e["tiff_path"]]): (h, e)
        for h, e in jobs
    }

    for i, future in enumerate(as_completed(futures), 1):
        h, e = futures[future]
        try:
            tiff_id, scene_record = future.result()
            with results_lock:
                scene_records[tiff_id] = scene_record
        except Exception as err:
            errors.append({"hour": h, "tiff_path": e["tiff_path"], "error": str(err)})

        if i % 10 == 0 or i == len(jobs):
            print(f"Processed {i}/{len(jobs)} | errors so far: {len(errors)}")

print(f"\nScenes processed: {len(scene_records)}")
print(f"Errors: {len(errors)}")

n_direct = sum(1 for s in scene_records.values() if s["grid_placement_method"] == "direct")
n_reprojected = sum(1 for s in scene_records.values() if s["grid_placement_method"] == "reprojected")
print(f"Sampled directly (no resampling): {n_direct}")
print(f"Reprojected (fallback): {n_reprojected}")
if n_reprojected:
    print("  *** Investigate these scenes -- they weren't on the expected native grid. ***")

if errors:
    print("\n=== Sampling Errors ===")
    for e in errors:
        print(f"\nHour: {e['hour']}")
        print(f"TIFF: {e['tiff_path']}")
        print(f"Error: {e['error']}")


# ================== Write single consolidated JSON ==================

manifest = {
    "step": "step5b_lst_grid_sample",
    "generated_at": datetime.now(ZoneInfo("UTC")).isoformat(),
    "grid_reference_path": grid_reference_path,   # now .shp, not .gpkg
    "master_grid_json_path": master_grid_json_path,
    "master_grid_id_raster_path": grid_id_raster_path,
    "grid_id_field": GRID_ID_FIELD,
    "grid_ids": GRID_ID_VALUES.tolist(),
    "n_grid_cells": N_GRID_CELLS,
    "lst_units": LST_UNITS,
    "sampling_method": "direct_pixel_sample",  # not zonal aggregation
    "native_res_m": NATIVE_RES,
    "day_start_hour": DAY_START_HOUR,
    "day_end_hour": DAY_END_HOUR,
    "round_decimals": ROUND_DECIMALS,
    "n_scenes": len(scene_records),
    "n_errors": len(errors),
    "n_scenes_direct": n_direct,
    "n_scenes_reprojected": n_reprojected,
    "errors": errors,
    "scenes": sorted(scene_records.values(), key=lambda e: e["datetime_local"]),
}

zonal_stats_json_path = os.path.join(zonal_stats_dir, "lst_zonal_stats.json")

print("\nStarting json dump")
with open(zonal_stats_json_path, "w") as f:
    json.dump(manifest, f, default=str)

print(f"\nStep 5B stats JSON saved -> {zonal_stats_json_path}")
print(f"Grid reference shapefile -> {grid_reference_path}")
print(f"Master grid metadata -> {master_grid_json_path}")
print(f"Master grid-id raster -> {grid_id_raster_path}")
print(f"Total scenes: {manifest['n_scenes']}")

Loaded Step 4 manifest -> /Users/ks/Desktop/Wu/Summer26/02_SOLA_2018_2025/masked_tiff/step5_grid_coverage_manifest.json
Workers: 10
LST output units: celsius
Tiffs to process: 219

Probing tiff headers to determine native LST grid...
CRS found across scenes: {'EPSG:32611': 219}
Resolutions found across scenes: {(70.0, 70.0): 219}
Master CRS: EPSG:32611
Native LST resolution: 70.0 m
Grid cell size (70 m) == native resolution -- direct sampling is valid.

Master grid: 1061 x 1402 px @ 70.0 m = 1061 x 1402 grid cells (1:1 with pixels)
Clipped grid to study boundary -- dropped 839859 cells outside boundary (647663 remain)
Grid cells: 647663
Grid reference shapefile saved -> /Users/ks/Desktop/Wu/Summer26/02_SOLA_2018_2025/lst_zonal_stats/grid_reference.shp
Master grid metadata saved -> /Users/ks/Desktop/Wu/Summer26/02_SOLA_2018_2025/lst_zonal_stats/master_grid.json
Grid-id raster saved -> /Users/ks/Desktop/Wu/Summer26/02_SOLA_2018_2025/lst_zonal_stats/master_grid_ids.tif
Processed 10/219 | 

# Diurnal Hourly Average LST by Season

In [2]:
# ============================================================
# Diurnal Hourly Average LST by Season -- Bar Plots (STANDALONE)
# Produces:
#   0. A JSON checkpoint of the (season, hour) averages used
#      to build the plots below -- <plot_dir>/diurnal_hourly_checkpoint.json
#   1. ONE combined plot -- 24 hour-groups x N bars/hour (1 per season)
#      -- bar labels show °F only
#   2. FOUR separate plots -- one per season, 24 bars (1 per hour)
#      -- bar labels show BOTH °C and °F
# Loads directly from lst_zonal_stats.json
# Outputs saved to <summary_dir>/diurnal_hourly_trends/
# ============================================================
import os
import json
import time
import numpy as np
import pandas as pd
from datetime import datetime
from zoneinfo import ZoneInfo
from matplotlib.figure import Figure
from IPython.display import display

CONFIG_PATH = "config.json"

SEASON_COLOR_MAP = {
    "winter": "#1f77b4",  # blue
    "spring": "#e377c2",  # pink
    "summer": "#d62728",  # red
    "fall": "#ff7f0e",    # orange
    "autumn": "#ff7f0e",  # orange (alias for fall)
}
FALLBACK_COLOR = "#7f7f7f"


def _find_season_keyword(season):
    """Extracts the base season keyword (winter/spring/summer/fall/autumn)
    from a season string that may have extra annotation, e.g.
    'Winter (DJF)' -> 'winter'. Returns None if no keyword is found."""
    s = season.strip().lower()
    for keyword in ("winter", "spring", "summer", "fall", "autumn"):
        if keyword in s:
            return keyword
    return None


def get_season_color(season):
    keyword = _find_season_keyword(season)
    color = SEASON_COLOR_MAP.get(keyword)
    if color is None:
        print(f"  *** WARNING: no color mapping for season '{season}' -- using fallback grey. "
              f"Check config season names against {list(SEASON_COLOR_MAP.keys())} ***")
        return FALLBACK_COLOR
    return color


# ---- Unit conversion helpers -- work regardless of source units ----

def to_celsius(value, units):
    return value if units == "celsius" else value - 273.15

def to_fahrenheit(value, units):
    return to_celsius(value, units) * 9 / 5 + 32


def load_hourly_season_data(config_path=CONFIG_PATH, verbose=True):
    """Loads zonal stats JSON and builds a (season, hour) -> avg lst_value
    table across all scenes and all grid cells."""
    t0 = time.time()
    def step(label):
        if verbose:
            print(f"  [{time.time()-t0:6.2f}s] {label}", flush=True)

    with open(config_path) as f:
        config = json.load(f)

    out_cfg = config["output"]
    root_dir = out_cfg.get("root_dir") or os.path.dirname(os.path.normpath(config["input_folder"]))
    zonal_stats_dir = os.path.join(root_dir, out_cfg.get("zonal_stats_dirname", "lst_zonal_stats"))
    summary_dir = os.path.join(root_dir, out_cfg.get("summary_dirname", "summary"))

    plot_dir = os.path.join(summary_dir, "diurnal_hourly_trends")
    os.makedirs(plot_dir, exist_ok=True)

    zonal_stats_json_path = os.path.join(zonal_stats_dir, "lst_zonal_stats.json")
    step(f"opening {zonal_stats_json_path} ({os.path.getsize(zonal_stats_json_path)/1e6:.1f} MB)")
    with open(zonal_stats_json_path) as f:
        zonal_stats = json.load(f)
    step("parsed")

    scenes = zonal_stats["scenes"]
    lst_units = zonal_stats.get("lst_units", "kelvin")
    step(f"scenes={len(scenes)}  units={lst_units}")
    if not scenes:
        raise ValueError(f"No scenes in {zonal_stats_json_path}")

    SEASON_ORDER = list(config["temporal_organization"]["season_months"].keys())

    step("building per-scene average lst_value (mean across grid cells)")
    rows = []
    for scene in scenes:
        vals = np.array(scene["lst_value"], dtype=float)
        valid = vals[~np.isnan(vals)]
        if valid.size == 0:
            continue
        rows.append({
            "season": scene["season"],
            "hour": int(scene["hour"]),
            "scene_avg": valid.mean(),
        })
    scene_df = pd.DataFrame(rows)
    step(f"scene_df built, {len(scene_df)} rows")

    step("aggregating to (season, hour) average across scenes")
    hourly_agg = (
        scene_df
        .groupby(["season", "hour"])["scene_avg"]
        .mean()
        .reset_index()
        .rename(columns={"scene_avg": "lst_avg"})
    )
    hourly_counts = (
        scene_df
        .groupby(["season", "hour"])["scene_avg"]
        .count()
        .reset_index()
        .rename(columns={"scene_avg": "n_scenes"})
    )
    hourly_agg = hourly_agg.merge(hourly_counts, on=["season", "hour"], how="left")
    step(f"hourly_agg built, {len(hourly_agg)} rows -- DONE in {time.time()-t0:.2f}s")

    return dict(
        hourly_agg=hourly_agg,
        SEASON_ORDER=SEASON_ORDER,
        plot_dir=plot_dir,
        lst_units=lst_units,
        zonal_stats_json_path=zonal_stats_json_path,
    )


def save_hourly_checkpoint(state, verbose=True):
    """Writes the (season, hour) -> avg LST table to a JSON checkpoint,
    so the exact numbers behind the bar plots below can be re-loaded
    without recomputing from the (much larger) raw zonal stats JSON.

    NOTE: day/night is NOT tagged yet upstream (intentionally omitted in
    Step 5B), so this checkpoint is hourly-only, not split by day/night.
    If day/night tagging is added back later, extend `rows` below to
    include it and re-run.
    """
    t0 = time.time()
    def step(label):
        if verbose:
            print(f"  [{time.time()-t0:6.2f}s] {label}", flush=True)

    step("START save_hourly_checkpoint")

    hourly_agg = state["hourly_agg"]
    seasons = state["SEASON_ORDER"]
    plot_dir = state["plot_dir"]
    lst_units = state["lst_units"]

    hours = list(range(24))
    rows = []
    for season in seasons:
        for h in hours:
            row = hourly_agg[(hourly_agg["season"] == season) & (hourly_agg["hour"] == h)]
            if row.empty:
                rows.append({
                    "season": season, "hour": h,
                    "lst_avg": None, "n_scenes": 0,
                })
            else:
                r = row.iloc[0]
                rows.append({
                    "season": season, "hour": h,
                    "lst_avg": round(float(r["lst_avg"]), 3),
                    "n_scenes": int(r["n_scenes"]),
                })
    step(f"built {len(rows)} (season, hour) rows")

    checkpoint = {
        "step": "diurnal_hourly_checkpoint",
        "generated_at": datetime.now(ZoneInfo("UTC")).isoformat(),
        "source_zonal_stats_json": state["zonal_stats_json_path"],
        "lst_units": lst_units,
        "season_order": seasons,
        "hours": hours,
        "day_night_tagged": False,  # explicit flag -- update if this changes upstream
        "records": rows,
    }

    checkpoint_path = os.path.join(plot_dir, "diurnal_hourly_checkpoint.json")
    step(f"writing -> {checkpoint_path}")
    with open(checkpoint_path, "w") as f:
        json.dump(checkpoint, f, indent=2)

    step(f"DONE in {time.time()-t0:.2f}s")
    return checkpoint_path


print("=" * 60)
print("LOADING DATA FOR DIURNAL HOURLY PLOTS")
print("=" * 60)
state = load_hourly_season_data()
print(f"\nData ready. Plots will be saved to -> {state['plot_dir']}")

print("\n" + "=" * 60)
print("SAVING CHECKPOINT")
print("=" * 60)
checkpoint_path = save_hourly_checkpoint(state)
print(f"Checkpoint saved -> {checkpoint_path}")


# ============================================================
# Plot 1: Combined -- all seasons on one chart
# Bar labels: °F only
# ============================================================

def plot_diurnal_hourly_trends_by_season(state, verbose=True):
    t0 = time.time()
    def step(label):
        if verbose:
            print(f"  [{time.time()-t0:6.2f}s] {label}", flush=True)

    step("START plot_diurnal_hourly_trends_by_season (combined)")

    hourly_agg = state["hourly_agg"]
    seasons = state["SEASON_ORDER"]
    plot_dir = state["plot_dir"]
    units = state["lst_units"]
    unit_label = "°C" if units == "celsius" else "K"

    hours = list(range(24))
    n_seasons = len(seasons)
    x = np.arange(len(hours))
    bar_width = 0.8 / n_seasons

    colors = {s: get_season_color(s) for s in seasons}

    fig = Figure(figsize=(22, 8))
    ax = fig.subplots(1, 1)

    for i, season in enumerate(seasons):
        step(f"building bars for season={season}")
        offset = (i - (n_seasons - 1) / 2) * bar_width
        vals = []
        for h in hours:
            row = hourly_agg[(hourly_agg["season"] == season) & (hourly_agg["hour"] == h)]
            vals.append(row["lst_avg"].iloc[0] if not row.empty else np.nan)
        vals = np.array(vals)

        ax.bar(x + offset, vals, width=bar_width, label=season, color=colors[season])

        for xi, v in zip(x + offset, vals):
            if np.isnan(v):
                continue
            f_val = to_fahrenheit(v, units)
            ax.text(xi, v, f"{f_val:.0f}°F", ha="center", va="bottom", fontsize=6, rotation=90)

    ax.set_xticks(x)
    ax.set_xticklabels([f"{h:02d}" for h in hours])
    ax.set_xlabel("Hour of day (local)")
    ax.set_ylabel(f"Average LST ({unit_label})")
    ax.set_title("Diurnal LST Trends by Season -- Average per Hour (0-23)", fontweight="bold")
    ax.legend(title="Season")
    ax.grid(axis="y", linestyle="--", alpha=0.4)

    step("running tight_layout")
    fig.tight_layout()

    out_path = os.path.join(plot_dir, "lst_diurnal_hourly_trends_by_season.png")
    step(f"saving -> {out_path}")
    fig.savefig(out_path, dpi=150, bbox_inches="tight")

    step(f"DONE in {time.time()-t0:.2f}s")
    return {"out_path": out_path, "fig": fig}


# ============================================================
# Plot 2: Separate -- one figure per season
# Bar labels: BOTH °C and °F
# ============================================================

def plot_diurnal_hourly_trend_single_season(season, state, color, verbose=True):
    t0 = time.time()
    def step(label):
        if verbose:
            print(f"    [{season} | {time.time()-t0:6.2f}s] {label}", flush=True)

    step("START plot_diurnal_hourly_trend_single_season")

    hourly_agg = state["hourly_agg"]
    plot_dir = state["plot_dir"]
    units = state["lst_units"]
    unit_label = "°C" if units == "celsius" else "K"

    season_df = hourly_agg[hourly_agg["season"] == season]
    if season_df.empty:
        step("SKIPPED -- no data for this season")
        return None

    hours = list(range(24))
    x = np.arange(len(hours))

    vals, n_scenes_list = [], []
    for h in hours:
        row = season_df[season_df["hour"] == h]
        if row.empty:
            vals.append(np.nan)
            n_scenes_list.append(0)
        else:
            vals.append(row["lst_avg"].iloc[0])
            n_scenes_list.append(int(row["n_scenes"].iloc[0]))
    vals = np.array(vals)

    fig = Figure(figsize=(14, 7))
    ax = fig.subplots(1, 1)

    ax.bar(x, vals, width=0.7, color=color)

    for xi, v, n in zip(x, vals, n_scenes_list):
        if np.isnan(v):
            continue
        c_val = to_celsius(v, units)
        f_val = to_fahrenheit(v, units)
        ax.text(xi, v, f"{c_val:.1f}°C\n{f_val:.1f}°F", ha="center", va="bottom",
                 fontsize=7, fontweight="bold", linespacing=1.3)

    ax.set_xticks(x)
    ax.set_xticklabels([f"{h:02d}" for h in hours])
    ax.set_xlabel("Hour of day (local)")
    ax.set_ylabel(f"Average LST ({unit_label})")
    ax.set_title(f"Diurnal LST Trend -- {season} -- Average per Hour (0-23)", fontweight="bold")
    ax.grid(axis="y", linestyle="--", alpha=0.4)
    ax.margins(y=0.15)  # extra headroom so 2-line labels don't get clipped

    step("running tight_layout")
    fig.tight_layout()

    safe_season = season.replace(" ", "_").replace("(", "").replace(")", "")
    out_path = os.path.join(plot_dir, f"lst_diurnal_hourly_trend_{safe_season}.png")
    step(f"saving -> {out_path}")
    fig.savefig(out_path, dpi=150, bbox_inches="tight")

    step(f"DONE in {time.time()-t0:.2f}s")
    return {"season": season, "out_path": out_path, "fig": fig}


print("\n" + "=" * 60)
print("PLOTTING: COMBINED")
print("=" * 60)
combined_result = plot_diurnal_hourly_trends_by_season(state)
print(f"\nSaved -> {combined_result['out_path']}")

print("\n" + "=" * 60)
print("PLOTTING: PER-SEASON (4 separate figures)")
print("=" * 60)

seasons = state["SEASON_ORDER"]

per_season_results = []
for season in seasons:
    print(f"\n--- {season} ---")
    color = get_season_color(season)
    result = plot_diurnal_hourly_trend_single_season(season, state, color)
    if result is not None:
        per_season_results.append(result)
        print(f"[{season}] COMPLETE -> {result['out_path']}")
    else:
        print(f"[{season}] skipped -- no data")

print("\n" + "=" * 60)
print("DISPLAY")
print("=" * 60)

display(combined_result["fig"])
for result in per_season_results:
    display(result["fig"])

print(f"\nCheckpoint JSON -> {checkpoint_path}")
print("Done.")

LOADING DATA FOR DIURNAL HOURLY PLOTS
  [  0.00s] opening /Users/ks/Desktop/Wu/Summer26/02_SOLA_2018_2025/lst_zonal_stats/lst_zonal_stats.json (980.4 MB)
  [ 17.91s] parsed
  [ 17.91s] scenes=219  units=celsius
  [ 17.91s] building per-scene average lst_value (mean across grid cells)
  [ 25.02s] scene_df built, 219 rows
  [ 25.02s] aggregating to (season, hour) average across scenes
  [ 25.03s] hourly_agg built, 71 rows -- DONE in 25.03s

Data ready. Plots will be saved to -> /Users/ks/Desktop/Wu/Summer26/02_SOLA_2018_2025/summary/diurnal_hourly_trends

SAVING CHECKPOINT
  [  0.00s] START save_hourly_checkpoint
  [  0.02s] built 96 (season, hour) rows
  [  0.02s] writing -> /Users/ks/Desktop/Wu/Summer26/02_SOLA_2018_2025/summary/diurnal_hourly_trends/diurnal_hourly_checkpoint.json
  [  0.03s] DONE in 0.03s
Checkpoint saved -> /Users/ks/Desktop/Wu/Summer26/02_SOLA_2018_2025/summary/diurnal_hourly_trends/diurnal_hourly_checkpoint.json

PLOTTING: COMBINED
  [  0.00s] START plot_diurnal_ho

<Figure size 2200x800 with 1 Axes>

<Figure size 1400x700 with 1 Axes>

<Figure size 1400x700 with 1 Axes>

<Figure size 1400x700 with 1 Axes>

<Figure size 1400x700 with 1 Axes>


Checkpoint JSON -> /Users/ks/Desktop/Wu/Summer26/02_SOLA_2018_2025/summary/diurnal_hourly_trends/diurnal_hourly_checkpoint.json
Done.


# Sample Size Plots -- Scenes & Valid Pixels per (Season, Hour)

In [3]:
# ============================================================
# Sample Size Plots -- Scenes & Valid Pixels per (Season, Hour)
# ------------------------------------------------------------
# Re-reads the RAW zonal stats JSON (not the checkpoint) to get
# per-scene valid-pixel counts, aggregates to (season, hour):
#   n_scenes        = number of distinct scenes in that bin
#   n_valid_pixels  = SUM of n_grid_cells_with_data across those
#                     scenes
#
# Layout: HORIZONTAL bars, hours on the y-axis (0 at top, 23 at
# bottom). Each PNG has 2 stacked subplots -- row 1 = scenes,
# row 2 = valid pixels -- so scene/pixel counts are visually
# separated but still travel together as one file. All text is
# black; font sizes bumped up throughout for readability.
#
# Produces:
#   1. ONE combined PNG -- grouped horizontal bars per hour,
#      one bar per season, row 1 = scenes, row 2 = pixels
#   2. FOUR separate per-season PNGs -- single horizontal bar
#      per hour, row 1 = scenes, row 2 = pixels
#
# Outputs saved to <summary_dir>/diurnal_hourly_trends/sample_size/
# ============================================================
import os
import json
import time
import numpy as np
import pandas as pd
from matplotlib.figure import Figure
from IPython.display import display

if "get_season_color" not in dir():
    raise RuntimeError(
        "This block depends on definitions from the diurnal hourly trends "
        "script (get_season_color, _find_season_keyword, CONFIG_PATH). "
        "Run that block first in this kernel."
    )

# ---- Shared font sizing (all text forced to black) ----
FONT = {
    "title": 18,
    "axis_label": 15,
    "tick": 12,
    "bar_label": 11,
    "legend": 12,
}
TEXT_COLOR = "black"


def load_sample_size_data(config_path=CONFIG_PATH, verbose=True):
    """Re-reads the RAW zonal stats JSON (not the checkpoint) to get
    per-scene valid-pixel counts, then aggregates to (season, hour)."""
    t0 = time.time()
    def step(label):
        if verbose:
            print(f"  [{time.time()-t0:6.2f}s] {label}", flush=True)

    with open(config_path) as f:
        config = json.load(f)

    out_cfg = config["output"]
    root_dir = out_cfg.get("root_dir") or os.path.dirname(os.path.normpath(config["input_folder"]))
    zonal_stats_dir = os.path.join(root_dir, out_cfg.get("zonal_stats_dirname", "lst_zonal_stats"))
    summary_dir = os.path.join(root_dir, out_cfg.get("summary_dirname", "summary"))
    diurnal_dir = os.path.join(summary_dir, "diurnal_hourly_trends")

    plot_dir = os.path.join(diurnal_dir, "sample_size")
    os.makedirs(plot_dir, exist_ok=True)

    zonal_stats_json_path = os.path.join(zonal_stats_dir, "lst_zonal_stats.json")
    step(f"opening {zonal_stats_json_path} ({os.path.getsize(zonal_stats_json_path)/1e6:.1f} MB)")
    with open(zonal_stats_json_path) as f:
        zonal_stats = json.load(f)
    step("parsed")

    scenes = zonal_stats["scenes"]
    if not scenes:
        raise ValueError(f"No scenes in {zonal_stats_json_path}")

    SEASON_ORDER = list(config["temporal_organization"]["season_months"].keys())

    step(f"building per-scene sample size rows from {len(scenes)} scenes")
    rows = []
    for scene in scenes:
        n_valid = scene.get("n_grid_cells_with_data")
        if n_valid is None:
            vals = scene.get("lst_value", [])
            n_valid = sum(1 for v in vals if v is not None)
        rows.append({
            "season": scene["season"],
            "hour": int(scene["hour"]),
            "tiff_id": scene["tiff_id"],
            "n_valid_pixels": int(n_valid),
        })
    scene_df = pd.DataFrame(rows)
    step(f"scene_df built, {len(scene_df)} rows")

    step("aggregating to (season, hour): n_scenes, total n_valid_pixels")
    sample_agg = (
        scene_df
        .groupby(["season", "hour"])
        .agg(n_scenes=("tiff_id", "nunique"), n_valid_pixels=("n_valid_pixels", "sum"))
        .reset_index()
    )
    step(f"sample_agg built, {len(sample_agg)} rows -- DONE in {time.time()-t0:.2f}s")

    return dict(
        sample_agg=sample_agg,
        SEASON_ORDER=SEASON_ORDER,
        plot_dir=plot_dir,
    )


print("=" * 60)
print("LOADING DATA FOR SAMPLE SIZE PLOTS")
print("=" * 60)
sample_state = load_sample_size_data()
print(f"\nData ready. Plots will be saved to -> {sample_state['plot_dir']}")


def _style_axis(ax, ylabel, xlabel=None, title=None):
    """Applies consistent black, larger-font styling to an axis."""
    if xlabel:
        ax.set_xlabel(xlabel, fontsize=FONT["axis_label"], color=TEXT_COLOR)
    ax.set_ylabel(ylabel, fontsize=FONT["axis_label"], color=TEXT_COLOR)
    if title:
        ax.set_title(title, fontsize=FONT["title"], fontweight="bold", color=TEXT_COLOR)
    ax.tick_params(axis="both", labelsize=FONT["tick"], colors=TEXT_COLOR)
    for spine_label in ax.spines:
        ax.spines[spine_label].set_color(TEXT_COLOR)
    ax.grid(axis="x", linestyle="--", alpha=0.4)


# ============================================================
# Plot 1: Combined -- grouped horizontal bars per hour, per season
# Row 1 = scenes, Row 2 = valid pixels
# ============================================================

def plot_sample_size_combined(sample_state, verbose=True):
    t0 = time.time()
    def step(label):
        if verbose:
            print(f"  [{time.time()-t0:6.2f}s] {label}", flush=True)

    step("START plot_sample_size_combined")

    sample_agg = sample_state["sample_agg"]
    seasons = sample_state["SEASON_ORDER"]
    plot_dir = sample_state["plot_dir"]

    hours = list(range(24))
    n_seasons = len(seasons)
    y = np.arange(len(hours))
    bar_height = 0.8 / n_seasons

    colors = {s: get_season_color(s) for s in seasons}

    fig = Figure(figsize=(14, 20))
    ax_scenes, ax_pixels = fig.subplots(2, 1)

    for i, season in enumerate(seasons):
        step(f"building bars for season={season}")
        offset = (i - (n_seasons - 1) / 2) * bar_height
        n_scenes_vals, n_pixels_vals = [], []
        for h in hours:
            row = sample_agg[(sample_agg["season"] == season) & (sample_agg["hour"] == h)]
            if row.empty:
                n_scenes_vals.append(0)
                n_pixels_vals.append(0)
            else:
                n_scenes_vals.append(int(row["n_scenes"].iloc[0]))
                n_pixels_vals.append(int(row["n_valid_pixels"].iloc[0]))
        n_scenes_vals = np.array(n_scenes_vals)
        n_pixels_vals = np.array(n_pixels_vals)

        ax_scenes.barh(y + offset, n_scenes_vals, height=bar_height, label=season, color=colors[season])
        ax_pixels.barh(y + offset, n_pixels_vals, height=bar_height, label=season, color=colors[season])

        for yi, v in zip(y + offset, n_scenes_vals):
            if v == 0:
                continue
            ax_scenes.text(v, yi, f" {int(v)}", ha="left", va="center",
                            fontsize=FONT["bar_label"], color=TEXT_COLOR)

        for yi, v in zip(y + offset, n_pixels_vals):
            if v == 0:
                continue
            ax_pixels.text(v, yi, f" {int(v):,}", ha="left", va="center",
                            fontsize=FONT["bar_label"], color=TEXT_COLOR)

    for ax in (ax_scenes, ax_pixels):
        ax.set_yticks(y)
        ax.set_yticklabels([f"{h:02d}" for h in hours], fontsize=FONT["tick"], color=TEXT_COLOR)
        ax.invert_yaxis()  # hour 0 at top
        ax.margins(x=0.15)

    _style_axis(ax_scenes, ylabel="Hour of day (local)", xlabel="Number of scenes",
                title="Sample Size by Season -- Scenes per Hour (0-23)")
    _style_axis(ax_pixels, ylabel="Hour of day (local)", xlabel="Number of valid pixels",
                title="Sample Size by Season -- Valid Pixels per Hour (0-23)")

    legend = ax_scenes.legend(title="Season", fontsize=FONT["legend"], loc="lower right")
    legend.get_title().set_fontsize(FONT["legend"])
    legend.get_title().set_color(TEXT_COLOR)
    for text in legend.get_texts():
        text.set_color(TEXT_COLOR)

    step("running tight_layout")
    fig.tight_layout()

    out_path = os.path.join(plot_dir, "lst_sample_size_combined.png")
    step(f"saving -> {out_path}")
    fig.savefig(out_path, dpi=150, bbox_inches="tight")

    step(f"DONE in {time.time()-t0:.2f}s")
    return {"out_path": out_path, "fig": fig}


# ============================================================
# Plot 2: Separate -- one PNG per season
# Row 1 = scenes, Row 2 = valid pixels
# ============================================================

def plot_sample_size_single_season(season, sample_state, color, verbose=True):
    t0 = time.time()
    def step(label):
        if verbose:
            print(f"    [{season} | {time.time()-t0:6.2f}s] {label}", flush=True)

    step("START plot_sample_size_single_season")

    sample_agg = sample_state["sample_agg"]
    plot_dir = sample_state["plot_dir"]

    season_df = sample_agg[sample_agg["season"] == season]
    if season_df.empty:
        step("SKIPPED -- no data for this season")
        return None

    hours = list(range(24))
    season_hourly = (
        season_df.set_index("hour")
        .reindex(hours, fill_value=0)
        .reset_index()
    )

    y = np.arange(len(hours))

    fig = Figure(figsize=(12, 16))
    ax_scenes, ax_pixels = fig.subplots(2, 1)

    ax_scenes.barh(y, season_hourly["n_scenes"], height=0.7, color=color)
    ax_pixels.barh(y, season_hourly["n_valid_pixels"], height=0.7, color=color)

    for yi, v in zip(y, season_hourly["n_scenes"]):
        if v == 0:
            continue
        ax_scenes.text(v, yi, f" {int(v)}", ha="left", va="center",
                        fontsize=FONT["bar_label"] + 1, fontweight="bold", color=TEXT_COLOR)

    for yi, v in zip(y, season_hourly["n_valid_pixels"]):
        if v == 0:
            continue
        ax_pixels.text(v, yi, f" {int(v):,}", ha="left", va="center",
                        fontsize=FONT["bar_label"] + 1, fontweight="bold", color=TEXT_COLOR)

    for ax in (ax_scenes, ax_pixels):
        ax.set_yticks(y)
        ax.set_yticklabels([f"{h:02d}" for h in hours], fontsize=FONT["tick"], color=TEXT_COLOR)
        ax.invert_yaxis()
        ax.margins(x=0.15)

    _style_axis(ax_scenes, ylabel="Hour of day (local)", xlabel="Number of scenes",
                title=f"Sample Size -- {season} -- Scenes per Hour (0-23)")
    _style_axis(ax_pixels, ylabel="Hour of day (local)", xlabel="Number of valid pixels",
                title=f"Sample Size -- {season} -- Valid Pixels per Hour (0-23)")

    step("running tight_layout")
    fig.tight_layout()

    safe_season = season.replace(" ", "_").replace("(", "").replace(")", "")
    out_path = os.path.join(plot_dir, f"lst_sample_size_{safe_season}.png")
    step(f"saving -> {out_path}")
    fig.savefig(out_path, dpi=150, bbox_inches="tight")

    step(f"DONE in {time.time()-t0:.2f}s")
    return {"season": season, "out_path": out_path, "fig": fig}


print("\n" + "=" * 60)
print("PLOTTING: SAMPLE SIZE COMBINED")
print("=" * 60)
sample_combined_result = plot_sample_size_combined(sample_state)
print(f"\nSaved -> {sample_combined_result['out_path']}")

print("\n" + "=" * 60)
print("PLOTTING: SAMPLE SIZE PER-SEASON (4 separate figures)")
print("=" * 60)

sample_seasons = sample_state["SEASON_ORDER"]
sample_per_season_results = []
for season in sample_seasons:
    print(f"\n--- {season} ---")
    color = get_season_color(season)
    result = plot_sample_size_single_season(season, sample_state, color)
    if result is not None:
        sample_per_season_results.append(result)
        print(f"[{season}] COMPLETE -> {result['out_path']}")
    else:
        print(f"[{season}] skipped -- no data")

print("\n" + "=" * 60)
print("DISPLAY")
print("=" * 60)

display(sample_combined_result["fig"])
for result in sample_per_season_results:
    display(result["fig"])

print("\nDone.")

LOADING DATA FOR SAMPLE SIZE PLOTS
  [  0.00s] opening /Users/ks/Desktop/Wu/Summer26/02_SOLA_2018_2025/lst_zonal_stats/lst_zonal_stats.json (980.4 MB)
  [ 11.82s] parsed
  [ 11.82s] building per-scene sample size rows from 219 scenes
  [ 16.56s] scene_df built, 219 rows
  [ 16.57s] aggregating to (season, hour): n_scenes, total n_valid_pixels
  [ 16.58s] sample_agg built, 71 rows -- DONE in 16.58s

Data ready. Plots will be saved to -> /Users/ks/Desktop/Wu/Summer26/02_SOLA_2018_2025/summary/diurnal_hourly_trends/sample_size

PLOTTING: SAMPLE SIZE COMBINED
  [  0.00s] START plot_sample_size_combined
  [  0.01s] building bars for season=Winter (DJF)
  [  0.03s] building bars for season=Spring (MAM)
  [  0.06s] building bars for season=Summer (JJA)
  [  0.08s] building bars for season=Fall (SON)
  [  0.13s] running tight_layout
  [  0.21s] saving -> /Users/ks/Desktop/Wu/Summer26/02_SOLA_2018_2025/summary/diurnal_hourly_trends/sample_size/lst_sample_size_combined.png
  [  0.63s] DONE in 0.

<Figure size 1400x2000 with 2 Axes>

<Figure size 1200x1600 with 2 Axes>

<Figure size 1200x1600 with 2 Axes>

<Figure size 1200x1600 with 2 Axes>

<Figure size 1200x1600 with 2 Axes>


Done.


# Seasonal Day/Night LST Summary

In [17]:
# # ============================================================
# # Seasonal Day/Night LST Summary -- Bar Plot (2 bars/season)
# # ------------------------------------------------------------
# # Loads the diurnal_hourly_checkpoint.json (produced by the
# # diurnal hourly trends script), tags each hour as day or night
# # using SEASON-SPECIFIC daylight windows, computes an n_scenes-
# # WEIGHTED average per (season, day_night), and plots 2 bars per
# # season with BOTH °C and °F labeled on each bar.
# #
# # Daylight windows used for tagging (local time):
# #   Winter (Dec-Feb): 7:00  AM - 5:00  PM
# #   Spring (Mar-May): 6:00  AM - 7:30  PM
# #   Summer (Jun-Aug): 5:45  AM - 8:00  PM
# #   Fall   (Sep-Nov): 6:30  AM - 6:00  PM
# #
# # Writes an UPDATED checkpoint (with day_night per record) and
# # the bar plot PNG, both into <plot_dir>.
# # ============================================================
# import os
# import json
# import time
# import numpy as np
# import pandas as pd
# from datetime import datetime
# from zoneinfo import ZoneInfo
# from matplotlib.figure import Figure
# from IPython.display import display
#
# CONFIG_PATH = "config.json"
#
# # ---- Season -> daylight window (decimal hours, local time) ----
# SEASON_DAYLIGHT_WINDOWS = {
#     "winter": (7.0, 17.0),    # 7:00 AM - 5:00 PM
#     "spring": (6.0, 19.5),    # 6:00 AM - 7:30 PM
#     "summer": (5.75, 20.0),   # 5:45 AM - 8:00 PM
#     "fall":   (6.5, 18.0),    # 6:30 AM - 6:00 PM
#     "autumn": (6.5, 18.0),    # alias for fall
# }
#
# DAY_NIGHT_COLORS = {"day": "#f4a259", "night": "#3d5a80"}
# DAY_NIGHT_ORDER = ["day", "night"]
#
#
# def _find_season_keyword(season):
#     """Extracts the base season keyword from a season string that may
#     have extra annotation, e.g. 'Winter (DJF)' -> 'winter'."""
#     s = season.strip().lower()
#     for keyword in ("winter", "spring", "summer", "fall", "autumn"):
#         if keyword in s:
#             return keyword
#     return None
#
#
# def get_daylight_window(season):
#     keyword = _find_season_keyword(season)
#     window = SEASON_DAYLIGHT_WINDOWS.get(keyword)
#     if window is None:
#         raise ValueError(
#             f"No daylight window defined for season '{season}' (parsed keyword: {keyword}). "
#             f"Known keywords: {list(SEASON_DAYLIGHT_WINDOWS.keys())}."
#         )
#     return window
#
#
# def classify_hour_day_night(season, hour):
#     start, end = get_daylight_window(season)
#     return "day" if start <= hour < end else "night"
#
#
# def to_celsius(value, units):
#     return value if units == "celsius" else value - 273.15
#
# def to_fahrenheit(value, units):
#     return to_celsius(value, units) * 9 / 5 + 32
#
#
# def load_checkpoint(config_path=CONFIG_PATH, verbose=True):
#     t0 = time.time()
#     def step(label):
#         if verbose:
#             print(f"  [{time.time()-t0:6.2f}s] {label}", flush=True)
#
#     with open(config_path) as f:
#         config = json.load(f)
#
#     out_cfg = config["output"]
#     root_dir = out_cfg.get("root_dir") or os.path.dirname(os.path.normpath(config["input_folder"]))
#     summary_dir = os.path.join(root_dir, out_cfg.get("summary_dirname", "summary"))
#     plot_dir = os.path.join(summary_dir, "diurnal_hourly_trends")
#
#     checkpoint_path = os.path.join(plot_dir, "diurnal_hourly_checkpoint.json")
#     step(f"opening checkpoint -> {checkpoint_path}")
#     with open(checkpoint_path) as f:
#         checkpoint = json.load(f)
#     step("parsed")
#
#     records_df = pd.DataFrame(checkpoint["records"])
#     step(f"records_df built, {len(records_df)} rows")
#
#     return dict(
#         checkpoint=checkpoint,
#         records_df=records_df,
#         SEASON_ORDER=checkpoint["season_order"],
#         lst_units=checkpoint["lst_units"],
#         plot_dir=plot_dir,
#         checkpoint_path=checkpoint_path,
#     )
#
#
# def tag_day_night(state, verbose=True):
#     """Adds a day_night column to records_df using per-season daylight
#     windows. Records with lst_avg == None are still tagged (for
#     completeness) but get dropped from averaging downstream."""
#     t0 = time.time()
#     def step(label):
#         if verbose:
#             print(f"  [{time.time()-t0:6.2f}s] {label}", flush=True)
#
#     step("START tag_day_night")
#     records_df = state["records_df"].copy()
#
#     records_df["day_night"] = records_df.apply(
#         lambda r: classify_hour_day_night(r["season"], r["hour"]), axis=1
#     )
#     step("tagged all records")
#
#     for season in state["SEASON_ORDER"]:
#         start, end = get_daylight_window(season)
#         step(f"  {season}: daylight window [{start}, {end}) -> "
#              f"day hours = {[h for h in range(24) if start <= h < end]}")
#
#     step(f"DONE tag_day_night in {time.time()-t0:.2f}s")
#     return records_df
#
#
# def save_tagged_checkpoint(state, records_df, verbose=True):
#     """Writes an UPDATED checkpoint including the day_night tag per record,
#     alongside the original (untagged) one -- doesn't overwrite it."""
#     t0 = time.time()
#     def step(label):
#         if verbose:
#             print(f"  [{time.time()-t0:6.2f}s] {label}", flush=True)
#
#     plot_dir = state["plot_dir"]
#     checkpoint = dict(state["checkpoint"])  # shallow copy
#     checkpoint["step"] = "diurnal_hourly_checkpoint_day_night_tagged"
#     checkpoint["generated_at"] = datetime.now(ZoneInfo("UTC")).isoformat()
#     checkpoint["day_night_tagged"] = True
#     checkpoint["daylight_windows"] = {
#         s: {"start_hour": get_daylight_window(s)[0],
#             "end_hour": get_daylight_window(s)[1]}
#         for s in state["SEASON_ORDER"]
#     }
#     checkpoint["records"] = records_df.replace({np.nan: None}).to_dict(orient="records")
#
#     out_path = os.path.join(plot_dir, "diurnal_hourly_checkpoint_day_night_tagged.json")
#     step(f"writing -> {out_path}")
#     with open(out_path, "w") as f:
#         json.dump(checkpoint, f, indent=2)
#
#     step(f"DONE in {time.time()-t0:.2f}s")
#     return out_path
#
#
# def compute_day_night_summary(records_df, seasons, verbose=True):
#     """n_scenes-WEIGHTED average lst_avg per (season, day_night)."""
#     t0 = time.time()
#     def step(label):
#         if verbose:
#             print(f"  [{time.time()-t0:6.2f}s] {label}", flush=True)
#
#     step("START compute_day_night_summary")
#     valid = records_df.dropna(subset=["lst_avg"]).copy()
#     valid = valid[valid["n_scenes"] > 0]
#
#     rows = []
#     for season in seasons:
#         season_df = valid[valid["season"] == season]
#         for day_night in DAY_NIGHT_ORDER:
#             subset = season_df[season_df["day_night"] == day_night]
#             if subset.empty:
#                 step(f"  {season}/{day_night}: no data, skipping")
#                 continue
#             weights = subset["n_scenes"].to_numpy()
#             vals = subset["lst_avg"].to_numpy()
#             weighted_avg = np.average(vals, weights=weights)
#             rows.append({
#                 "season": season,
#                 "day_night": day_night,
#                 "avg": weighted_avg,
#                 "min": vals.min(),
#                 "max": vals.max(),
#                 "n_scenes_total": int(weights.sum()),
#                 "n_hours": len(subset),
#             })
#             step(f"  {season}/{day_night}: weighted_avg={weighted_avg:.2f} "
#                  f"(n_hours={len(subset)}, n_scenes_total={int(weights.sum())})")
#
#     summary_df = pd.DataFrame(rows)
#     step(f"DONE compute_day_night_summary -- {len(summary_df)} rows in {time.time()-t0:.2f}s")
#     return summary_df
#
#
# print("=" * 60)
# print("LOADING CHECKPOINT")
# print("=" * 60)
# state = load_checkpoint()
# print(f"Loaded from -> {state['checkpoint_path']}")
#
# print("\n" + "=" * 60)
# print("TAGGING DAY/NIGHT (season-specific windows)")
# print("=" * 60)
# records_df = tag_day_night(state)
#
# print("\n" + "=" * 60)
# print("SAVING DAY/NIGHT-TAGGED CHECKPOINT")
# print("=" * 60)
# tagged_checkpoint_path = save_tagged_checkpoint(state, records_df)
# print(f"Saved -> {tagged_checkpoint_path}")
#
# print("\n" + "=" * 60)
# print("COMPUTING DAY/NIGHT SUMMARY (n_scenes-weighted)")
# print("=" * 60)
# seasons = state["SEASON_ORDER"]
# summary_df = compute_day_night_summary(records_df, seasons)
# units = state["lst_units"]
# unit_label = "°C" if units == "celsius" else "K"
#
# print("\n" + "=" * 60)
# print(f"NUMERIC SUMMARY TABLE (weighted avg LST, {unit_label})")
# print("=" * 60)
# print(f"{'Season':<15} {'Day/Night':<10} {'Avg':>10} {'Min':>10} {'Max':>10} {'N scenes':>10}")
# print("-" * 68)
# for _, r in summary_df.iterrows():
#     print(f"{r['season']:<15} {r['day_night']:<10} {r['avg']:>10.2f} {r['min']:>10.2f} "
#           f"{r['max']:>10.2f} {r['n_scenes_total']:>10}")
# print("=" * 68)
#
#
# # ============================================================
# # Plot: 2 bars per season (day, night), colored by day/night
# # Labels show BOTH °C and °F on each bar
# # ============================================================
#
# fig = Figure(figsize=(12, 7))
# ax = fig.subplots(1, 1)
#
# n_seasons = len(seasons)
# x = np.arange(n_seasons)
# bar_width = 0.35
#
# for i, day_night in enumerate(DAY_NIGHT_ORDER):
#     offset = (i - 0.5) * bar_width
#     avgs = []
#     for season in seasons:
#         row = summary_df[(summary_df["season"] == season) & (summary_df["day_night"] == day_night)]
#         avgs.append(row["avg"].iloc[0] if not row.empty else np.nan)
#     avgs = np.array(avgs)
#
#     ax.bar(
#         x + offset, avgs, width=bar_width,
#         label=day_night.capitalize(), color=DAY_NIGHT_COLORS[day_night],
#     )
#
#     for xi, avg in zip(x + offset, avgs):
#         if np.isnan(avg):
#             continue
#         c_val = to_celsius(avg, units)
#         f_val = to_fahrenheit(avg, units)
#         ax.text(xi, avg, f"{c_val:.1f}°C\n{f_val:.1f}°F", ha="center", va="bottom",
#                  fontsize=8, fontweight="bold", linespacing=1.3)
#
# ax.set_xticks(x)
# ax.set_xticklabels(seasons)
# ax.set_ylabel(f"LST Average ({unit_label})")
# ax.set_title("Seasonal LST Summary -- Day vs Night (season-specific daylight windows)",
#              fontweight="bold")
# ax.legend(title="Day/Night")
# ax.grid(axis="y", linestyle="--", alpha=0.4)
# ax.margins(y=0.15)  # extra headroom so 2-line labels don't get clipped
#
# fig.tight_layout()
#
# out_path = os.path.join(state["plot_dir"], "lst_seasonal_day_night_bar_chart.png")
# fig.savefig(out_path, dpi=150, bbox_inches="tight")
# print(f"\nSaved -> {out_path}")
#
# display(fig)
#
# print("\nDone.")

LOADING CHECKPOINT
  [  0.00s] opening checkpoint -> /Users/ks/Desktop/Wu/Summer26/02_SOLA_2018_2025/summary/diurnal_hourly_trends/diurnal_hourly_checkpoint.json
  [  0.00s] parsed
  [  0.00s] records_df built, 96 rows
Loaded from -> /Users/ks/Desktop/Wu/Summer26/02_SOLA_2018_2025/summary/diurnal_hourly_trends/diurnal_hourly_checkpoint.json

TAGGING DAY/NIGHT (season-specific windows)
  [  0.00s] START tag_day_night
  [  0.00s] tagged all records
  [  0.00s]   Winter (DJF): daylight window [7.0, 17.0) -> day hours = [7, 8, 9, 10, 11, 12, 13, 14, 15, 16]
  [  0.00s]   Spring (MAM): daylight window [6.0, 19.5) -> day hours = [6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]
  [  0.00s]   Summer (JJA): daylight window [5.75, 20.0) -> day hours = [6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]
  [  0.00s]   Fall (SON): daylight window [6.5, 18.0) -> day hours = [7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17]
  [  0.00s] DONE tag_day_night in 0.00s

SAVING DAY/NIGHT-TAGGED CHECKPOINT
  [ 

<Figure size 1200x700 with 1 Axes>


Done.


# STOP HERE TILL NIGHT & DAY = CLARIFIED
Step 6A: Load and Aggregate Zonal Statistics

In [ ]:
# # ============================================================
# # CHUNK A: Load inputs + build long_df / agg
# # Run this first. Produces `state` dict used by Chunk B.
# # ============================================================
# import os
# import json
# import time
# import numpy as np
# import pandas as pd
# import geopandas as gpd
#
# STAT_COLS = ["lst_mean", "lst_median"]  # std removed
#
#
# def load_pipeline_inputs(config_path="config.json", verbose=True):
#     """Loads everything needed straight from disk -- fully self-contained."""
#     t0 = time.time()
#     def step(label):
#         if verbose:
#             print(f"  [{time.time()-t0:7.2f}s] {label}", flush=True)
#
#     step("START load_pipeline_inputs")
#
#     step(f"opening config: {config_path}")
#     with open(config_path) as f:
#         config = json.load(f)
#     step("config loaded")
#
#     out_cfg = config["output"]
#     grid_cfg = config["grid_coverage"]
#     GRID_ID_FIELD = grid_cfg["grid_id_field"]
#
#     root_dir = out_cfg.get("root_dir") or os.path.dirname(os.path.normpath(config["input_folder"]))
#     zonal_stats_dir = os.path.join(root_dir, out_cfg.get("zonal_stats_dirname", "lst_zonal_stats"))
#     summary_dir = os.path.join(root_dir, out_cfg.get("summary_dirname", "summary"))
#     os.makedirs(summary_dir, exist_ok=True)
#     step(f"paths resolved -> zonal_stats_dir={zonal_stats_dir}")
#
#     zonal_stats_json_path = os.path.join(zonal_stats_dir, "lst_zonal_stats.json")
#     step(f"opening zonal stats JSON ({os.path.getsize(zonal_stats_json_path)/1e6:.1f} MB) -- this can be slow")
#     with open(zonal_stats_json_path) as f:
#         zonal_stats = json.load(f)
#     step("zonal stats JSON parsed")
#
#     grid_ids = zonal_stats["grid_ids"]
#     scenes = zonal_stats["scenes"]
#     step(f"grid_ids={len(grid_ids)}  scenes={len(scenes)}")
#     if not scenes:
#         raise ValueError(f"No scenes in {zonal_stats_json_path} -- check Step 5B ran successfully.")
#
#     grid_reference_path = zonal_stats.get("grid_reference_path") or os.path.join(
#         zonal_stats_dir, "grid_reference.gpkg"
#     )
#     step(f"reading grid geometry from {grid_reference_path} -- this can be slow for large gpkg files")
#     grid_geom = gpd.read_file(grid_reference_path)
#     step(f"grid_geom loaded, {len(grid_geom)} rows")
#
#     step("checking for duplicate grid IDs (blowup check)")
#     n_unique_ids = len(np.unique(grid_ids))
#     n_unique_geom = grid_geom[GRID_ID_FIELD].nunique()
#     step(f"grid_ids: {len(grid_ids)} total, {n_unique_ids} unique | "
#          f"grid_geom: {len(grid_geom)} total, {n_unique_geom} unique {GRID_ID_FIELD}")
#     if n_unique_ids != len(grid_ids) or n_unique_geom != len(grid_geom):
#         step("  *** WARNING: duplicate IDs detected -- merges downstream may blow up rows ***")
#
#     step(f"building long_df from {len(scenes)} scenes -- looping now")
#     frames = []
#     for i, scene in enumerate(scenes):
#         frames.append(pd.DataFrame({
#             GRID_ID_FIELD: grid_ids,
#             "lst_mean": scene["lst_mean"],
#             "lst_median": scene["lst_median"],
#             "day_night": scene["day_night"],
#             "season": scene["season"],
#         }))
#         if verbose and (i + 1) % 25 == 0:
#             step(f"  ...built {i+1}/{len(scenes)} scene frames")
#     step(f"all {len(scenes)} scene frames built, concatenating")
#     long_df = pd.concat(frames, ignore_index=True)
#     step(f"long_df concatenated, {len(long_df)} rows")
#
#     step("running groupby aggregation -- this can be slow if IDs are duplicated")
#     agg = (
#         long_df
#         .groupby(["season", "day_night", GRID_ID_FIELD])[STAT_COLS]
#         .mean()
#         .reset_index()
#     )
#     step(f"aggregation done, agg has {len(agg)} rows")
#
#     SEASON_ORDER = list(config["temporal_organization"]["season_months"].keys())
#     step(f"season order: {SEASON_ORDER}")
#
#     scene_counts = (
#         pd.DataFrame(scenes)
#         .groupby(["season", "day_night"])["tiff_id"]
#         .nunique()
#         .to_dict()
#     )
#     step("scene_counts built")
#
#     step(f"DONE load_pipeline_inputs -- total {time.time()-t0:.2f}s")
#     if verbose:
#         print(f"[load_pipeline_inputs] SUMMARY: "
#               f"scenes={len(scenes)}  grid_ids={len(grid_ids)}  "
#               f"grid_geom_rows={len(grid_geom)}  long_df_rows={len(long_df)}  agg_rows={len(agg)}")
#
#     return dict(
#         config=config, GRID_ID_FIELD=GRID_ID_FIELD, root_dir=root_dir,
#         zonal_stats_dir=zonal_stats_dir, summary_dir=summary_dir,
#         zonal_stats=zonal_stats, grid_ids=grid_ids, scenes=scenes,
#         grid_geom=grid_geom, long_df=long_df, agg=agg,
#         SEASON_ORDER=SEASON_ORDER, scene_counts=scene_counts,
#     )
#
#
# print("=" * 60)
# print("CHUNK A: LOADING PIPELINE INPUTS + BUILDING long_df / agg")
# print("=" * 60)
# state = load_pipeline_inputs()
# print("\nChunk A complete. `state` is ready for Chunk B.")

# Step 6B: Plot Seasonal LST Statistics Maps

In [ ]:
# # ============================================================
# # CHUNK B: Plotting (sequential, no threading)
# # Run this after Chunk A. If `state` isn't defined (fresh
# # kernel / standalone run), it loads inputs itself first.
# # - lst_std column removed (mean + median only)
# # - Uniform color scale across ALL plots: vmin=200, vmax=400 (Kelvin)
# # - Verbose print statements at every step to catch hangs
# # ============================================================
# import os
# import time
# from matplotlib.figure import Figure
# from IPython.display import display
#
# # ---- Fixed uniform legend range (applies to every subplot) ----
# VMIN = 200
# VMAX = 400
#
# STAT_COLS = ["lst_mean", "lst_median"]
# STAT_LABELS = {"lst_mean": "Mean LST", "lst_median": "Median LST"}
# STAT_CMAPS = {"lst_mean": "inferno", "lst_median": "inferno"}
# DAY_NIGHT_ORDER = ["day", "night"]
#
# # Standalone safety: if Chunk A wasn't run in this session, run it now.
# if "state" not in dir():
#     print("`state` not found -- running load_pipeline_inputs() now (Chunk A wasn't run first).")
#     from_chunk_a_missing = True
#     # NOTE: requires load_pipeline_inputs() to be defined (i.e. Chunk A's
#     # function definition must have executed at least once in this kernel).
#     state = load_pipeline_inputs()
#
#
# def build_season_figure(season, state, verbose=True):
#     """Builds and saves ONE season's figure. Sequential -- no threading."""
#     t0 = time.time()
#     def step(label):
#         if verbose:
#             print(f"    [{season} | {time.time()-t0:6.2f}s] {label}", flush=True)
#
#     agg = state["agg"]
#     grid_geom = state["grid_geom"]
#     GRID_ID_FIELD = state["GRID_ID_FIELD"]
#     scene_counts = state["scene_counts"]
#     summary_dir = state["summary_dir"]
#
#     step("START build_season_figure")
#
#     season_df = agg[agg["season"] == season]
#     if season_df.empty:
#         step("SKIPPED -- no scenes for this season")
#         return f"[{season}] skipped -- no scenes"
#
#     step("creating Figure object")
#     fig = Figure(figsize=(12, 12))
#     fig.suptitle(f"LST Zonal Statistics -- {season}", fontsize=16, fontweight="bold")
#     axes = fig.subplots(2, 2)
#     step("Figure + axes created")
#
#     for row_i, day_night in enumerate(DAY_NIGHT_ORDER):
#         step(f"row {row_i}: filtering subset for day_night={day_night}")
#         subset = season_df[season_df["day_night"] == day_night]
#         step(f"subset rows: {len(subset)} -- starting merge with grid_geom")
#
#         merge_t0 = time.time()
#         plot_gdf = grid_geom.merge(subset, on=GRID_ID_FIELD, how="left")
#         step(f"merge done in {time.time()-merge_t0:.2f}s -- plot_gdf rows: {len(plot_gdf)}")
#         if len(plot_gdf) > len(grid_geom):
#             step(f"  *** WARNING: merge blew up ({len(plot_gdf)} > {len(grid_geom)} grid cells) ***")
#
#         n_scenes = scene_counts.get((season, day_night), 0)
#
#         for col_i, stat_col in enumerate(STAT_COLS):
#             step(f"  drawing subplot [{day_night}/{stat_col}]")
#             ax = axes[row_i, col_i]
#
#             if plot_gdf[stat_col].notna().sum() == 0:
#                 ax.set_title(f"{day_night.capitalize()} -- {STAT_LABELS[stat_col]}\n(no data)")
#                 ax.axis("off")
#                 step(f"  subplot [{day_night}/{stat_col}] has no data -- skipped")
#                 continue
#
#             plot_t0 = time.time()
#             plot_gdf.plot(
#                 column=stat_col,
#                 ax=ax,
#                 legend=True,
#                 cmap=STAT_CMAPS[stat_col],
#                 edgecolor="none",
#                 vmin=VMIN,
#                 vmax=VMAX,
#                 missing_kwds={"color": "lightgrey", "label": "No data"},
#                 legend_kwds={"label": f"{STAT_LABELS[stat_col]} (K)", "shrink": 0.7},
#             )
#             step(f"  subplot [{day_night}/{stat_col}] plotted in {time.time()-plot_t0:.2f}s")
#
#             ax.set_title(f"{day_night.capitalize()} -- {STAT_LABELS[stat_col]} (n={n_scenes})")
#             ax.set_xticks([])
#             ax.set_yticks([])
#             ax.set_aspect("equal")
#
#     step("running tight_layout")
#     fig.tight_layout(rect=[0, 0, 1, 0.96])
#     step("tight_layout done")
#
#     safe_season = season.replace(" ", "_").replace("(", "").replace(")", "")
#     out_path = os.path.join(summary_dir, f"lst_zonal_grid_{safe_season}.png")
#
#     step(f"starting savefig -> {out_path}")
#     save_t0 = time.time()
#     fig.savefig(out_path, dpi=150, bbox_inches="tight")
#     step(f"savefig done in {time.time()-save_t0:.2f}s")
#
#     elapsed = time.time() - t0
#     step(f"DONE build_season_figure -- total {elapsed:.2f}s")
#     return {"season": season, "out_path": out_path, "elapsed": elapsed, "fig": fig}
#
#
# print("=" * 60)
# print("CHUNK B: PLOTTING")
# print("=" * 60)
# print(f"Seasons to process: {state['SEASON_ORDER']}\n")
#
# results = []
# t_all = time.time()
#
# for i, season in enumerate(state["SEASON_ORDER"]):
#     print("=" * 60)
#     print(f"SEASON {i+1}/{len(state['SEASON_ORDER'])}: {season}")
#     print("=" * 60)
#     try:
#         result = build_season_figure(season, state)
#         if isinstance(result, dict):
#             print(f"[{result['season']}] COMPLETE in {result['elapsed']:.2f}s -> {result['out_path']}\n")
#             results.append(result)
#         else:
#             print(f"{result}\n")
#     except Exception as e:
#         print(f"[{season}] FAILED: {e}\n")
#
# print("=" * 60)
# print(f"ALL SEASONS DONE -- total {time.time()-t_all:.2f}s")
# print("=" * 60)
#
# for result in results:
#     display(result["fig"])
#
# print("\nDone.")

# Step 6C: Plot Seasonal Mean LST Maps

In [ ]:
# # ============================================================
# # CHUNK B: Plotting (sequential, no threading)
# # Run this after Chunk A. If `state` isn't defined (fresh
# # kernel / standalone run), it loads inputs itself first.
# # - ONLY lst_mean is plotted (median removed)
# # - Legend is uniform WITHIN each season (day vs night share the
# #   same vmin/vmax, computed from that season's own min/max),
# #   but each season gets its own scale so seasons aren't forced
# #   onto one global range.
# # - Verbose print statements at every step to catch hangs
# # ============================================================
# import os
# import time
# from matplotlib.figure import Figure
# from IPython.display import display
#
# STAT_COLS = ["lst_mean"]
# STAT_LABELS = {"lst_mean": "Mean LST"}
# STAT_CMAPS = {"lst_mean": "inferno"}
# DAY_NIGHT_ORDER = ["day", "night"]
#
# # Standalone safety: if Chunk A wasn't run in this session, run it now.
# if "state" not in dir():
#     print("`state` not found -- running load_pipeline_inputs() now (Chunk A wasn't run first).")
#     state = load_pipeline_inputs()
#
#
# def build_season_figure(season, state, verbose=True):
#     """Builds and saves ONE season's figure. Sequential -- no threading.
#     Color scale (vmin/vmax) is computed per-season from that season's own
#     lst_mean values across BOTH day and night, so the legend is uniform
#     within the season (day vs night comparable) but not across seasons."""
#     t0 = time.time()
#     def step(label):
#         if verbose:
#             print(f"    [{season} | {time.time()-t0:6.2f}s] {label}", flush=True)
#
#     agg = state["agg"]
#     grid_geom = state["grid_geom"]
#     GRID_ID_FIELD = state["GRID_ID_FIELD"]
#     scene_counts = state["scene_counts"]
#     summary_dir = state["summary_dir"]
#
#     step("START build_season_figure")
#
#     season_df = agg[agg["season"] == season]
#     if season_df.empty:
#         step("SKIPPED -- no scenes for this season")
#         return f"[{season}] skipped -- no scenes"
#
#     step("computing per-season vmin/vmax from lst_mean (day + night combined)")
#     season_vmin = season_df["lst_mean"].min()
#     season_vmax = season_df["lst_mean"].max()
#     step(f"season_vmin={season_vmin:.2f}  season_vmax={season_vmax:.2f}")
#
#     step("creating Figure object")
#     fig = Figure(figsize=(6, 12))
#     fig.suptitle(f"LST Mean -- {season}", fontsize=16, fontweight="bold")
#     axes = fig.subplots(2, 1)
#     step("Figure + axes created")
#
#     for row_i, day_night in enumerate(DAY_NIGHT_ORDER):
#         step(f"row {row_i}: filtering subset for day_night={day_night}")
#         subset = season_df[season_df["day_night"] == day_night]
#         step(f"subset rows: {len(subset)} -- starting merge with grid_geom")
#
#         merge_t0 = time.time()
#         plot_gdf = grid_geom.merge(subset, on=GRID_ID_FIELD, how="left")
#         step(f"merge done in {time.time()-merge_t0:.2f}s -- plot_gdf rows: {len(plot_gdf)}")
#         if len(plot_gdf) > len(grid_geom):
#             step(f"  *** WARNING: merge blew up ({len(plot_gdf)} > {len(grid_geom)} grid cells) ***")
#
#         n_scenes = scene_counts.get((season, day_night), 0)
#         ax = axes[row_i]
#
#         if plot_gdf["lst_mean"].notna().sum() == 0:
#             ax.set_title(f"{day_night.capitalize()} -- Mean LST\n(no data)")
#             ax.axis("off")
#             step(f"  subplot [{day_night}/lst_mean] has no data -- skipped")
#             continue
#
#         step(f"  drawing subplot [{day_night}/lst_mean]")
#         plot_t0 = time.time()
#         plot_gdf.plot(
#             column="lst_mean",
#             ax=ax,
#             legend=True,
#             cmap=STAT_CMAPS["lst_mean"],
#             edgecolor="none",
#             vmin=season_vmin,   # <-- uniform within this season only
#             vmax=season_vmax,   # <-- uniform within this season only
#             missing_kwds={"color": "lightgrey", "label": "No data"},
#             legend_kwds={"label": "Mean LST (K)", "shrink": 0.7},
#         )
#         step(f"  subplot [{day_night}/lst_mean] plotted in {time.time()-plot_t0:.2f}s")
#
#         ax.set_title(f"{day_night.capitalize()} -- Mean LST (n={n_scenes})")
#         ax.set_xticks([])
#         ax.set_yticks([])
#         ax.set_aspect("equal")
#
#     step("running tight_layout")
#     fig.tight_layout(rect=[0, 0, 1, 0.96])
#     step("tight_layout done")
#
#     safe_season = season.replace(" ", "_").replace("(", "").replace(")", "")
#     out_path = os.path.join(summary_dir, f"lst_zonal_grid_{safe_season}.png")
#
#     step(f"starting savefig -> {out_path}")
#     save_t0 = time.time()
#     fig.savefig(out_path, dpi=150, bbox_inches="tight")
#     step(f"savefig done in {time.time()-save_t0:.2f}s")
#
#     elapsed = time.time() - t0
#     step(f"DONE build_season_figure -- total {elapsed:.2f}s")
#     return {
#         "season": season, "out_path": out_path, "elapsed": elapsed, "fig": fig,
#         "vmin": season_vmin, "vmax": season_vmax,
#     }
#
#
# print("=" * 60)
# print("CHUNK B: PLOTTING")
# print("=" * 60)
# print(f"Seasons to process: {state['SEASON_ORDER']}\n")
#
# results = []
# t_all = time.time()
#
# for i, season in enumerate(state["SEASON_ORDER"]):
#     print("=" * 60)
#     print(f"SEASON {i+1}/{len(state['SEASON_ORDER'])}: {season}")
#     print("=" * 60)
#     try:
#         result = build_season_figure(season, state)
#         if isinstance(result, dict):
#             print(f"[{result['season']}] COMPLETE in {result['elapsed']:.2f}s "
#                   f"(scale {result['vmin']:.1f}-{result['vmax']:.1f}K) -> {result['out_path']}\n")
#             results.append(result)
#         else:
#             print(f"{result}\n")
#     except Exception as e:
#         print(f"[{season}] FAILED: {e}\n")
#
# print("=" * 60)
# print(f"ALL SEASONS DONE -- total {time.time()-t_all:.2f}s")
# print("=" * 60)
#
# for result in results:
#     display(result["fig"])
#
# print("\nDone.")

# Step 6D: Plot Seasonal Mean LST Summary

In [ ]:
# # ============================================================
# # CHUNK C: Summary Stats Bar Chart (avg + min/max range bars)
# # Run after Chunk A (needs `state`). Standalone-safe: loads
# # inputs itself if `state` isn't defined in this kernel.
# # - 4 season groups x 2 bars (day/night)
# # - Bar height = average lst_mean across the grid
# # - Error bars show min -> max range across the grid
# # - Only the AVERAGE value is labeled numerically on each bar
# #   (min/max are still printed in the table below, just not
# #   labeled on the chart itself)
# # - Exports PNG to <root_dir>/summary/
# # ============================================================
# import os
# import time
# import numpy as np
# import pandas as pd
# from matplotlib.figure import Figure
# from IPython.display import display
#
# # Standalone safety: if Chunk A wasn't run in this session, run it now.
# if "state" not in dir():
#     print("`state` not found -- running load_pipeline_inputs() now (Chunk A wasn't run first).")
#     state = load_pipeline_inputs()
#
# DAY_NIGHT_ORDER = ["day", "night"]
# DAY_NIGHT_COLORS = {"day": "#f4a259", "night": "#3d5a80"}
#
#
# def compute_summary_stats(state, verbose=True):
#     """For each (season, day_night), compute avg / min / max of lst_mean
#     across all grid cells. Returns a tidy DataFrame."""
#     t0 = time.time()
#     def step(label):
#         if verbose:
#             print(f"  [{time.time()-t0:6.2f}s] {label}", flush=True)
#
#     step("START compute_summary_stats")
#     agg = state["agg"]
#     scene_counts = state["scene_counts"]
#
#     rows = []
#     for season in state["SEASON_ORDER"]:
#         season_df = agg[agg["season"] == season]
#         if season_df.empty:
#             step(f"  {season}: no data, skipping")
#             continue
#         for day_night in DAY_NIGHT_ORDER:
#             subset = season_df[season_df["day_night"] == day_night]
#             vals = subset["lst_mean"].dropna()
#             if vals.empty:
#                 step(f"  {season}/{day_night}: no data, skipping")
#                 continue
#             rows.append({
#                 "season": season,
#                 "day_night": day_night,
#                 "avg": vals.mean(),
#                 "min": vals.min(),
#                 "max": vals.max(),
#                 "n_scenes": scene_counts.get((season, day_night), 0),
#                 "n_cells": len(vals),
#             })
#             step(f"  {season}/{day_night}: avg={vals.mean():.2f}K  min={vals.min():.2f}K  max={vals.max():.2f}K")
#
#     summary_df = pd.DataFrame(rows)
#     step(f"DONE compute_summary_stats -- {len(summary_df)} rows in {time.time()-t0:.2f}s")
#     return summary_df
#
#
# print("=" * 60)
# print("CHUNK C: SUMMARY STATS")
# print("=" * 60)
#
# summary_df = compute_summary_stats(state)
#
# # ---- Printed numeric table ----
# print("\n" + "=" * 60)
# print("NUMERIC SUMMARY TABLE (Mean LST, Kelvin)")
# print("=" * 60)
# print(f"{'Season':<12} {'Day/Night':<10} {'Avg (K)':>10} {'Min (K)':>10} {'Max (K)':>10} {'Range':>8}")
# print("-" * 62)
# for _, r in summary_df.iterrows():
#     print(f"{r['season']:<12} {r['day_night']:<10} {r['avg']:>10.2f} {r['min']:>10.2f} "
#           f"{r['max']:>10.2f} {r['max']-r['min']:>8.2f}")
# print("=" * 62)
#
# # ---- Bar chart: 4 season groups x 2 bars (day/night) ----
# seasons = state["SEASON_ORDER"]
# n_seasons = len(seasons)
# x = np.arange(n_seasons)
# bar_width = 0.35
#
# fig = Figure(figsize=(12, 7))
# ax = fig.subplots(1, 1)
#
# for i, day_night in enumerate(DAY_NIGHT_ORDER):
#     offset = (i - 0.5) * bar_width
#     avgs, mins, maxs = [], [], []
#     for season in seasons:
#         row = summary_df[(summary_df["season"] == season) & (summary_df["day_night"] == day_night)]
#         if row.empty:
#             avgs.append(np.nan); mins.append(np.nan); maxs.append(np.nan)
#         else:
#             r = row.iloc[0]
#             avgs.append(r["avg"]); mins.append(r["min"]); maxs.append(r["max"])
#
#     avgs, mins, maxs = (
#         np.array(avgs),
#         np.array(mins),
#         np.array(maxs),
#     )
#     lower_err = avgs - mins
#     upper_err = maxs - avgs
#
#     bars = ax.bar(
#         x + offset, avgs, width=bar_width,
#         label=day_night.capitalize(), color=DAY_NIGHT_COLORS[day_night],
#         yerr=[lower_err, upper_err], capsize=5, ecolor="black",
#     )
#
#     # ---- Numeric label: AVERAGE only ----
#     for xi, avg, upper in zip(x + offset, avgs, upper_err):
#         if np.isnan(avg):
#             continue
#         ax.text(xi, avg + upper + 2, f"{avg:.1f}K", ha="center", va="bottom",
#                  fontsize=9, fontweight="bold")
#
# ax.set_xticks(x)
# ax.set_xticklabels(seasons)
# ax.set_ylabel("LST Mean (K)")
# ax.set_title("Seasonal LST Summary -- Average with Min/Max Range (Day vs Night)", fontweight="bold")
# ax.legend(title="Day/Night")
# ax.grid(axis="y", linestyle="--", alpha=0.4)
#
# fig.tight_layout()
#
# # ---- Export to summary_dir ----
# out_path = os.path.join(state["summary_dir"], "lst_seasonal_summary_bar_chart.png")
# fig.savefig(out_path, dpi=150, bbox_inches="tight")
# print(f"\nSaved -> {out_path}")
#
# display(fig)
#
# print("\nDone.")

In [ ]:
print('Done')